In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             log_loss, confusion_matrix, classification_report,
                             roc_curve, auc, cohen_kappa_score, matthews_corrcoef)
from sklearn.preprocessing import label_binarize
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define Project Paths
BASE_DIR = "/content/drive/MyDrive/AI_Image_Project"
FEATURES_DIR = os.path.join(BASE_DIR, "features")
MODELS_DIR = os.path.join(BASE_DIR, "saved_models")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
CONF_DIR = os.path.join(RESULTS_DIR, "confusion_matrices")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(CONF_DIR, exist_ok=True)

print("Loading optimized 500-feature matrices and strict split indices...")

# 3. Load Features, Labels, and Splits
X = np.load(os.path.join(FEATURES_DIR, "X_combined_500.npy"))
y = np.load(os.path.join(FEATURES_DIR, "y_labels_15k.npy"))

train_idx = np.load(os.path.join(FEATURES_DIR, "train_idx.npy"))
val_idx   = np.load(os.path.join(FEATURES_DIR, "val_idx.npy"))
test_idx  = np.load(os.path.join(FEATURES_DIR, "test_idx.npy"))

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val     = X[val_idx], y[val_idx]
X_test, y_test   = X[test_idx], y[test_idx]

CLASS_NAMES = np.array(["Authentic", "Full AI", "Partial AI"])
print(f"Data ready. Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading optimized 500-feature matrices and strict split indices...
Data ready. Train: (12000, 500) | Val: (1500, 500) | Test: (1500, 500)


In [ ]:
# ===== Cell 2: Normalization + PCA =====
scaler_path = os.path.join(FEATURES_DIR, "scaler_params.npz")
if os.path.exists(scaler_path):
    scaler = np.load(scaler_path)
    mu, sigma = scaler["mu"], scaler["sigma"]
    print("Loaded training scaler parameters.")
else:
    mu = np.mean(X_train, axis=0)
    sigma = np.std(X_train, axis=0)
    sigma[sigma < 1e-8] = 1.0
    print("Calculated new scaler parameters.")

epsilon = 1e-8
X_train_norm = (X_train - mu) / (sigma + epsilon)
X_val_norm   = (X_val - mu) / (sigma + epsilon)
X_test_norm  = (X_test - mu) / (sigma + epsilon)
print(f"Normalization verified. Train Mean: {np.mean(X_train_norm):.4f}, Train Std: {np.std(X_train_norm):.4f}")


class ScratchPCA:
    """
    PCA from scratch via SVD of the centered training matrix.
    Fit on TRAIN only (no leakage). Components are sorted by explained variance,
    so keeping the first k columns of the projection == PCA with k components.
    """
    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        Xc = X - self.mean_
        U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
        self.components_ = Vt                                  # (n_features, n_features)
        self.explained_variance_ = (S ** 2) / (X.shape[0] - 1)
        self.explained_variance_ratio_ = self.explained_variance_ / self.explained_variance_.sum()
        return self

    def transform(self, X, k):
        return (X - self.mean_) @ self.components_[:k].T


pca = ScratchPCA().fit(X_train_norm)
cum_var = np.cumsum(pca.explained_variance_ratio_)
for target in [0.80, 0.90, 0.95, 0.99]:
    print(f"Components needed for {int(target*100)}% variance: {int(np.searchsorted(cum_var, target) + 1)}")

plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, len(cum_var) + 1), cum_var, lw=2)
plt.axhline(0.95, color='r', ls='--', lw=1, label='95% variance')
plt.title("PCA Cumulative Explained Variance (fit on train)")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
class RandomFourierFeatures:
    """
    Approximates the RBF (Gaussian) kernel using Random Fourier Features.
    Maps input data to a higher-dimensional space where linear SGD can learn non-linear boundaries.
    """
    def __init__(self, n_components=1000, gamma=0.1, random_state=42):
        self.n_components = n_components
        self.gamma = gamma
        self.random_state = random_state
        self.W = None
        self.b = None

    def fit(self, n_features):
        rng = np.random.default_rng(self.random_state)
        # W drawn from a Gaussian distribution with variance 2*gamma
        self.W = rng.normal(0, np.sqrt(2 * self.gamma), (n_features, self.n_components))
        # b drawn from a Uniform distribution between 0 and 2*pi
        self.b = rng.uniform(0, 2 * np.pi, self.n_components)
        return self

    def transform(self, X):
        # Projection: sqrt(2/D) * cos(XW + b)
        Z = np.dot(X, self.W) + self.b
        return np.sqrt(2.0 / self.n_components) * np.cos(Z)

class ScratchRBFSVM_OvR:
    """
    Non-Linear SVM from scratch using Random Fourier Features and OvR Mini-Batch SGD.
    Includes proper n_samples scaling for L2 regularization to prevent gradient explosion.
    """
    def __init__(self, C=1.0, gamma=0.1, n_components=1000, learning_rate=0.05, epochs=150, batch_size=256, decay=0.001, random_state=42):
        self.C = float(C)
        self.gamma = float(gamma)
        self.n_components = int(n_components)
        self.lr = float(learning_rate)
        self.epochs = int(epochs)
        self.batch_size = int(batch_size)
        self.decay = float(decay)
        self.random_state = random_state

        self.rff = None
        self.models = []
        self.classes_ = None
        self.train_loss_history = []
        self.val_loss_history = []

    def _compute_hinge_loss(self, X_mapped, y_binary, W, b, lambda_reg):
        n_samples = X_mapped.shape[0]
        distances = y_binary * (np.dot(X_mapped, W) + b)
        distances[distances > 1] = 1 # max(0, 1 - y*(wx+b))
        hinge = np.mean(1 - distances)
        # Scale L2 loss by n_samples to match gradient and prevent overflow
        l2 = (lambda_reg / (2 * n_samples)) * np.sum(W ** 2)
        return hinge + l2

    def fit(self, X, y, X_val=None, y_val=None, verbose=False):
        n_samples, n_features = X.shape
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)

        # 1. Fit RFF mapping and transform features
        self.rff = RandomFourierFeatures(n_components=self.n_components, gamma=self.gamma, random_state=self.random_state)
        self.rff.fit(n_features)

        X_mapped = self.rff.transform(X)
        if X_val is not None:
            X_val_mapped = self.rff.transform(X_val)

        lambda_reg = 1.0 / max(self.C, 1e-12)
        rng = np.random.default_rng(self.random_state)

        # Initialize W and b for each OvR model
        self.models = [{'W': rng.normal(0, 0.01, self.n_components), 'b': 0.0} for _ in range(n_classes)]
        y_ovr = np.where(y[:, None] == self.classes_, 1, -1)
        if X_val is not None:
            y_val_ovr = np.where(y_val[:, None] == self.classes_, 1, -1)

        step = 0
        for epoch in range(self.epochs):
            perm = rng.permutation(n_samples)
            X_shuffled = X_mapped[perm]
            y_ovr_shuffled = y_ovr[perm]

            for start_idx in range(0, n_samples, self.batch_size):
                end_idx = min(start_idx + self.batch_size, n_samples)
                X_batch = X_shuffled[start_idx:end_idx]
                y_batch_ovr = y_ovr_shuffled[start_idx:end_idx]
                b_size = X_batch.shape[0]

                current_lr = self.lr / (1.0 + self.decay * step)
                step += 1

                for k in range(n_classes):
                    W = self.models[k]['W']
                    b = self.models[k]['b']
                    y_b = y_batch_ovr[:, k]

                    distances = 1 - y_b * (np.dot(X_batch, W) + b)
                    mask = (distances > 0).astype(float)

                    # FIXED: Scaled L2 gradient by n_samples to prevent explosion
                    dW = (lambda_reg / n_samples) * W - np.dot(X_batch.T, y_b * mask) / b_size
                    db = -np.sum(y_b * mask) / b_size

                    self.models[k]['W'] -= current_lr * dW
                    self.models[k]['b'] -= current_lr * db

            if verbose and (epoch % 20 == 0 or epoch == self.epochs - 1):
                t_loss = sum(self._compute_hinge_loss(X_mapped, y_ovr[:, k], self.models[k]['W'], self.models[k]['b'], lambda_reg) for k in range(n_classes))
                self.train_loss_history.append(t_loss)
                if X_val is not None:
                    v_loss = sum(self._compute_hinge_loss(X_val_mapped, y_val_ovr[:, k], self.models[k]['W'], self.models[k]['b'], lambda_reg) for k in range(n_classes))
                    self.val_loss_history.append(v_loss)

        return self

    def decision_function(self, X):
        X_mapped = self.rff.transform(X)
        scores = np.zeros((X.shape[0], len(self.classes_)))
        for k in range(len(self.classes_)):
            scores[:, k] = np.dot(X_mapped, self.models[k]['W']) + self.models[k]['b']
        return scores

    def predict_proba(self, X):
        # NOTE: Uncalibrated pseudo-probabilities derived via a basic sigmoid
        # mapping of the margin distances. Standard SVM output is technically non-probabilistic.
        scores = self.decision_function(X)
        sigmoids = 1.0 / (1.0 + np.exp(-np.clip(scores, -50, 50)))
        return sigmoids / np.sum(sigmoids, axis=1, keepdims=True)

    def predict(self, X):
        scores = self.decision_function(X)
        return self.classes_[np.argmax(scores, axis=1)]

print("RBF-SVM via Random Fourier Features Initialized.")

In [ ]:
# ===== Cell 4: Tuning (n_components, C, gamma) =====
k_candidates = [50, 100, 200, 300]
C_candidates = [1.0, 10.0]
gamma_candidates = [0.0001, 0.0005, 0.001, 0.005]
tuning_results = []

# Pre-project once per k (projection onto first k components of the same SVD)
proj = {k: (pca.transform(X_train_norm, k), pca.transform(X_val_norm, k)) for k in k_candidates}

print("Starting PCA + RBF Hyperparameter Tuning Grid Search...")
for k in k_candidates:
    Xtr_k, Xva_k = proj[k]
    for c_val in C_candidates:
        for g_val in gamma_candidates:
            t0 = time.time()
            temp_model = ScratchRBFSVM_OvR(C=c_val, gamma=g_val, n_components=1000, epochs=150, learning_rate=0.05)

            with np.errstate(all='ignore'):
                temp_model.fit(Xtr_k, y_train)

            if any(np.isnan(m['W']).any() for m in temp_model.models):
                print(f"k={k:<4} | C={c_val:<5} | Gamma={g_val:<7} | Skipped due to divergence/overflow.")
                continue

            val_f1 = f1_score(y_val, temp_model.predict(Xva_k), average='macro')
            elapsed = time.time() - t0
            tuning_results.append({"k": k, "C": c_val, "Gamma": g_val, "Val_F1": val_f1, "Time (s)": elapsed})
            print(f"k={k:<4} | C={c_val:<5} | Gamma={g_val:<7} | Val Macro-F1: {val_f1:.4f} | Time: {elapsed:.2f}s")

df_tune = pd.DataFrame(tuning_results)
best_config = df_tune.sort_values(by="Val_F1", ascending=False).iloc[0]
best_k, best_c, best_gamma = int(best_config["k"]), best_config["C"], best_config["Gamma"]

print("\n" + "=" * 40)
print(f"Optimal Configuration: k = {best_k}, C = {best_c}, Gamma = {best_gamma}")
print(f"PCA variance retained at k={best_k}: {cum_var[best_k-1]*100:.2f}%")
print("=" * 40)

if (best_k in [min(k_candidates), max(k_candidates)] or
    best_c in [min(C_candidates), max(C_candidates)] or
    best_gamma in [min(gamma_candidates), max(gamma_candidates)]):
    print("⚠️ WARNING: A best parameter lies at the edge of the search grid. Consider expanding the grid limits.")

# Heatmap: Gamma vs k (best Val F1 over C)
pivot_table = df_tune.groupby(["k", "Gamma"])["Val_F1"].max().reset_index().pivot(index="k", columns="Gamma", values="Val_F1")
plt.figure(figsize=(8, 5))
sns.heatmap(pivot_table, annot=True, fmt=".4f", cmap="viridis")
plt.title("PCA + RBF-SVM Validation Macro-F1 (best over C)")
plt.show()

In [ ]:
# ===== Cell 5: Final training =====
X_train_p = pca.transform(X_train_norm, best_k)
X_val_p   = pca.transform(X_val_norm, best_k)
X_test_p  = pca.transform(X_test_norm, best_k)

print(f"Training Final PCA({best_k}) + RBF-SVM with C={best_c}, Gamma={best_gamma}, RFF=2000...")
final_rbf_svm = ScratchRBFSVM_OvR(
    C=best_c,
    gamma=best_gamma,
    n_components=2000,
    epochs=300,
    learning_rate=0.05
)

start_t = time.time()
final_rbf_svm.fit(X_train_p, y_train, X_val=X_val_p, y_val=y_val, verbose=True)
train_time = time.time() - start_t
print(f"Final training finished in {train_time:.2f} seconds.")

plt.figure(figsize=(8, 4))
plt.plot(final_rbf_svm.train_loss_history, label='Train Hinge Loss', lw=2)
plt.plot(final_rbf_svm.val_loss_history, label='Val Hinge Loss', lw=2, linestyle='--')
plt.title("PCA + RBF-SVM Convergence (OvR Hinge Loss + L2)")
plt.xlabel("Checkpoint (every 20 epochs)")
plt.ylabel("Cumulative Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ===== Cell 6: Test evaluation =====
# Single test evaluation. Do not tune after this.
print("=" * 60)
print("FINAL TEST EVALUATION METRICS (PCA + RBF-SVM)")
print("=" * 60)

y_test_pred = final_rbf_svm.predict(X_test_p)
y_test_prob = final_rbf_svm.predict_proba(X_test_p)
y_test_scores = final_rbf_svm.decision_function(X_test_p)
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])

auc_scores_ovr = [roc_auc_score(y_test_bin[:, k], y_test_scores[:, k]) for k in range(len(CLASS_NAMES))]
macro_roc_auc = np.mean(auc_scores_ovr)

metrics_dict = {
    "Model": "RBF-SVM + PCA (Scratch RFF OvR)",
    "Accuracy (%)": round(accuracy_score(y_test, y_test_pred) * 100, 2),
    "Macro F1": round(f1_score(y_test, y_test_pred, average='macro'), 4),
    "Weighted F1": round(f1_score(y_test, y_test_pred, average='weighted'), 4),
    "ROC-AUC (OvR)": round(macro_roc_auc, 4),
    "Log Loss": round(log_loss(y_test, y_test_prob), 4),
    "Cohen's Kappa": round(cohen_kappa_score(y_test, y_test_pred), 4),
    "MCC": round(matthews_corrcoef(y_test, y_test_pred), 4),
    "Train Time (s)": round(train_time, 2)
}

# NOTE: Log Loss is from uncalibrated pseudo-probabilities (not comparable to Logistic Regression).
for k_, v in metrics_dict.items():
    print(f"{k_:<20}: {v}")
print(f"{'PCA components':<20}: {best_k} ({cum_var[best_k-1]*100:.2f}% variance)")
print("\n" + classification_report(y_test, y_test_pred, target_names=CLASS_NAMES))

plt.figure(figsize=(8, 6))
colors = ['#2ecc71', '#e74c3c', '#f1c40f']
for i, color in enumerate(colors):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_test_scores[:, i])
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{CLASS_NAMES[i]} (AUC = {auc_scores_ovr[i]:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.title("PCA + RBF-SVM ROC Curve (One-vs-Rest)", fontsize=14)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.show()

In [ ]:
# ===== Cell 7: Confusion matrix =====
cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cbar=False)
plt.title(f"PCA + RBF-SVM Confusion Matrix (Acc: {metrics_dict['Accuracy (%)']}%)")
plt.ylabel("True Class")
plt.xlabel("Predicted Class")
plt.tight_layout()
plt.savefig(os.path.join(CONF_DIR, "svm_rbf_pca_cm_15k.png"), dpi=200)
plt.show()

In [ ]:
# ===== Cell 8: Save model, probs, metrics =====
model_save_path = os.path.join(MODELS_DIR, "svm_rbf_pca_model.npz")
W_all = np.array([m['W'] for m in final_rbf_svm.models])
b_all = np.array([m['b'] for m in final_rbf_svm.models])

np.savez(model_save_path,
         W=W_all,
         b=b_all,
         rff_W=final_rbf_svm.rff.W,
         rff_b=final_rbf_svm.rff.b,
         pca_mean=pca.mean_,
         pca_components=pca.components_[:best_k],   # (k, 500)
         best_k=best_k,
         best_c=best_c,
         best_gamma=best_gamma,
         best_lr=final_rbf_svm.lr,
         class_names=CLASS_NAMES)
print(f"PCA + RBF-SVM parameters saved to: {model_save_path}")

np.save(os.path.join(MODELS_DIR, "svm_rbf_pca_val_probs.npy"), final_rbf_svm.predict_proba(X_val_p))
np.save(os.path.join(MODELS_DIR, "svm_rbf_pca_test_probs.npy"), y_test_prob)

csv_path = os.path.join(RESULTS_DIR, "model_comparison_15k.csv")
if os.path.exists(csv_path):
    df_results = pd.read_csv(csv_path)
    df_results = df_results[df_results["Model"] != metrics_dict["Model"]]
    df_results = pd.concat([df_results, pd.DataFrame([metrics_dict])], ignore_index=True)
else:
    df_results = pd.DataFrame([metrics_dict])
df_results.to_csv(csv_path, index=False)
print(f"Metrics appended to: {csv_path}")

In [ ]:
# ===== Cell 9: Standalone prediction check =====
def predict_rbf_pca_from_saved(path, X_normalized):
    """
    Loads saved parameters and reproduces predictions: PCA -> RFF -> linear OvR scores.
    X_normalized must be z-scored with the training scaler (500 features).
    """
    data = np.load(path, allow_pickle=True)
    W, b = data['W'], data['b']
    rff_W, rff_b = data['rff_W'], data['rff_b']
    pca_mean, pca_comp = data['pca_mean'], data['pca_components']

    X_p = (X_normalized - pca_mean) @ pca_comp.T                 # PCA projection
    n_components = rff_W.shape[1]
    X_mapped = np.sqrt(2.0 / n_components) * np.cos(X_p @ rff_W + rff_b)
    decision_scores = X_mapped @ W.T + b
    return np.argmax(decision_scores, axis=1), decision_scores

# Do not tune after this point.
print("Testing standalone prediction function on test set...")
saved_preds, saved_scores = predict_rbf_pca_from_saved(model_save_path, X_test_norm)
live_preds = final_rbf_svm.predict(X_test_p)
assert np.array_equal(saved_preds, live_preds), "Mismatch between saved model and live model!"
print("Success! Standalone function predictions match the live trained PCA + RBF model perfectly.")